<a href="https://colab.research.google.com/github/HoGentTIN/ml-workshop-middelbaar-2627/blob/main/notebooks/CNN_SteenPapierSchaar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steen, Papier, Schaar met een CNN

In de MNIST-notebook heb je een CNN gebouwd dat handgeschreven cijfers herkent.
Nu pas je dat zelf toe op een nieuwe dataset: kleurenfoto's van een hand die **steen**, **papier** of **schaar** toont.

| Deel | Onderwerp |
|---|---|
| A | De data inlezen en klaarmaken |
| B | Zelf een CNN bouwen en trainen |
| C | Het model evalueren |
| D | Uitdaging: maak het model beter & test met je eigen hand! |

Voer de cellen uit met **Shift + Enter**. In cellen met `# TODO` schrijf je zelf de code: kijk gerust terug naar je MNIST-notebook!
Zet eerst een GPU aan: **Runtime → Runtimetype wijzigen → T4 GPU**.

---
## Deel A: De data

De dataset **Rock Paper Scissors** van Google bevat 2520 trainings- en 372 testafbeeldingen.
De code hieronder leest ze in en verkleint ze naar 100 × 100 pixels. Je krijgt dezelfde variabelen als bij MNIST:
`x_train`, `y_train`, `x_test` en `y_test`. Het label is `0` = steen, `1` = papier, `2` = schaar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers

GROOTTE = 100
KLASSEN = ["steen", "papier", "schaar"]

def laad(split):
    ds = tfds.load("rock_paper_scissors", split=split, as_supervised=True)
    ds = ds.map(lambda x, y: (tf.image.resize(x, (GROOTTE, GROOTTE)), y))
    x, y = next(iter(ds.batch(10_000)))
    return x.numpy(), y.numpy()

x_train, y_train = laad("train")
x_test, y_test = laad("test")
print("Trainingsset:", x_train.shape, " Testset:", x_test.shape)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].astype("uint8"))
    plt.title(KLASSEN[y_train[i]])
    plt.axis("off")
plt.show()

**Opdracht:** normaliseer de pixelwaarden naar het bereik 0–1, net als bij MNIST.

Bij MNIST moest je ook een kanaal-as toevoegen met `[..., None]`. Is dat hier nog nodig? Kijk naar de `shape` hierboven.

In [ ]:
# TODO: normaliseer x_train en x_test
x_train = ...
x_test = ...
print("Vorm:", x_train.shape, " min:", x_train.min(), " max:", x_train.max())

---
## Deel B: Een CNN bouwen en trainen

**Opdracht:** bouw een CNN voor deze dataset. Vertrek van je MNIST-model en pas aan wat nodig is:
- de **input** is nu 100 × 100 × 3;
- er zijn maar **3** klassen;
- de afbeeldingen zijn groter en ingewikkelder: gebruik **drie** blokken `Conv2D` + `MaxPooling2D`, met 16, 32 en 64 filters.

In [ ]:
# TODO: bouw het CNN
model = keras.Sequential([
    ...
])
model.summary()

Hoeveel parameters (`Total params`) heeft je model? Een gewoon netwerk met één `Dense(128)`-laag rechtstreeks op de
100 × 100 × 3 = 30 000 pixels heeft er al 30 000 × 128 ≈ **3,8 miljoen**. Waarom heeft een CNN er zoveel minder nodig?

**Opdracht:** compileer het model en train het **10** epochs. Gebruik de testset als `validation_data` en bewaar het resultaat in `geschiedenis`.

In [ ]:
# TODO: compileer en train het model
model.compile(...)
geschiedenis = model.fit(...)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(geschiedenis.history["accuracy"], "o-", label="training")
plt.plot(geschiedenis.history["val_accuracy"], "o-", label="test")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Bij MNIST lagen training en test dicht bij elkaar. Hier wordt de training-accuracy (bijna) 100%, maar de test-accuracy blijft lager.
Hoe noemen we dat? Waarom gebeurt het hier wel en bij MNIST niet?

---
## Deel C: Evalueren

**Opdracht:** net als bij MNIST:
1. bereken de accuracy op de testset;
2. voorspel de klasse van elke testafbeelding (`voorspeld`);
3. toon de confusion matrix (gebruik `display_labels=KLASSEN`).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# TODO: accuracy, voorspellingen en confusion matrix
_, acc = ...
print(f"Accuracy op de testset: {acc:.1%}")

voorspeld = ...

...
plt.show()

In [ ]:
fout = np.where(voorspeld != y_test)[0]
print(len(fout), "fouten op", len(y_test), "testafbeeldingen")

plt.figure(figsize=(12, 5))
for n, i in enumerate(fout[:10]):
    plt.subplot(2, 5, n + 1)
    plt.imshow(x_test[i])
    plt.title(f"echt: {KLASSEN[y_test[i]]}\nzegt: {KLASSEN[voorspeld[i]]}", color="red", fontsize=9)
    plt.axis("off")
plt.show()

Welke klassen worden het vaakst verward? Zie je bij de fouten een reden?

---
## Deel D: Uitdaging

### D1. Een beter model

Twee trucs tegen overfitting:
- **Data-augmentatie:** de trainingsafbeeldingen bij elke epoch willekeurig spiegelen, draaien en inzoomen.
  Zo lijkt het alsof we veel meer verschillende afbeeldingen hebben.
- **Dropout:** tijdens het trainen willekeurig een deel van de neuronen uitzetten, zodat het netwerk niet van enkele neuronen afhankelijk wordt.

In [ ]:
augmentatie = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

plt.figure(figsize=(10, 2.5))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(augmentatie(x_test[0:1], training=True)[0])
    plt.axis("off")
plt.suptitle("Dezelfde afbeelding, 5 keer willekeurig aangepast")
plt.show()

**Opdracht:** bouw een verbeterd CNN en haal een zo hoog mogelijke test-accuracy. Ideeën:
- zet `augmentatie` als eerste laag (net na `keras.Input`);
- voeg `layers.Dropout(0.25)` toe vóór de `Dense`-lagen;
- meer of minder filters, een extra `Conv2D`-laag, meer epochs, ...

Wie haalt de hoogste test-accuracy?

In [ ]:
mijn_model = keras.Sequential([
    keras.Input(shape=(GROOTTE, GROOTTE, 3)),
    # TODO: bouw je eigen, verbeterde CNN
    ...
])

mijn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mijn_geschiedenis = mijn_model.fit(x_train, y_train, epochs=15, batch_size=32, validation_data=(x_test, y_test))

plt.plot(mijn_geschiedenis.history["accuracy"], "o-", label="training")
plt.plot(mijn_geschiedenis.history["val_accuracy"], "o-", label="test")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### D2. Test met je eigen hand

Neem een foto van je hand (steen, papier of schaar), liefst **tegen een witte achtergrond**, en upload ze hieronder.

In [ ]:
from PIL import Image

try:
    from google.colab import files
    upload = files.upload()
    bestanden = list(upload.keys())
except ImportError:
    print("Uploaden werkt enkel in Google Colab.")
    bestanden = []

for bestand in bestanden:
    foto = Image.open(bestand).convert("RGB").resize((GROOTTE, GROOTTE))
    foto = np.array(foto) / 255.0
    kans = mijn_model.predict(foto[None], verbose=0)[0]

    plt.imshow(foto)
    plt.title(" | ".join(f"{k}: {p:.0%}" for k, p in zip(KLASSEN, kans)))
    plt.axis("off")
    plt.show()

Werkt het met jouw foto? De trainingsafbeeldingen zijn **computergegenereerde** handen op een witte achtergrond.
Wat betekent dat voor foto's uit de echte wereld?

> Een model is maar zo goed als de data waarmee het getraind is!